# 00 — Load Data
Downloads four Kaggle pharmacy/pharma datasets to `../data/` using `opendatasets`.
You will be prompted once for your **Kaggle username** and **API key**  
(get the key from https://www.kaggle.com/settings → *API* → *Create New Token*).

In [37]:
%pip install -q kaggle pandas python-dotenv openpyxl json5 dotenv lxml

Note: you may need to restart the kernel to use updated packages.


In [38]:
import os
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Load credentials from ../.env into the environment BEFORE importing kaggle.
# The kaggle client authenticates on import using KAGGLE_USERNAME / KAGGLE_KEY.
load_dotenv(Path("../.env"))

assert os.getenv("KAGGLE_USERNAME"), "KAGGLE_USERNAME missing — set it in .env"
assert os.getenv("KAGGLE_KEY"),      "KAGGLE_KEY missing — set it in .env"

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# (owner/dataset-slug, target subfolder)
DATASETS = [
    "milanzdravkovic/pharma-sales-data",
    "pritipoddar/inventory-data-for-pharmacy-website-in-json-format",
    "mohammedashraf000/pharmaceutical-supply-chain-optimization",
    "hossam82/pharmacy-products-dataset",
    "gpreda/covid-world-vaccination-progress",
    "arashnic/flu-data",
]

print(f"Authenticated as: {os.getenv('KAGGLE_USERNAME')}")
print(f"Data directory:   {DATA_DIR.resolve()}")

Authenticated as: jetanin
Data directory:   C:\Work\LogisticsInnovationHackathon2026\VaxFlow\data


In [39]:
def download(slug: str, base_dir: Path) -> Path:
    """Download + unzip a Kaggle dataset into base_dir/<name>/. Skips if present."""
    name = slug.split("/")[1]
    dest = base_dir / name
    if dest.exists() and any(dest.iterdir()):
        print(f"[skip] {slug} — already in {dest}")
        return dest
    dest.mkdir(parents=True, exist_ok=True)
    print(f"[download] {slug}")
    api.dataset_download_files(slug, path=str(dest), unzip=True, quiet=False)
    return dest


for slug in DATASETS:
    download(slug, DATA_DIR)

print("\nAll datasets downloaded to:", DATA_DIR.resolve())

[skip] milanzdravkovic/pharma-sales-data — already in ..\data\pharma-sales-data
[skip] pritipoddar/inventory-data-for-pharmacy-website-in-json-format — already in ..\data\inventory-data-for-pharmacy-website-in-json-format
[skip] mohammedashraf000/pharmaceutical-supply-chain-optimization — already in ..\data\pharmaceutical-supply-chain-optimization
[skip] hossam82/pharmacy-products-dataset — already in ..\data\pharmacy-products-dataset
[skip] gpreda/covid-world-vaccination-progress — already in ..\data\covid-world-vaccination-progress
[skip] arashnic/flu-data — already in ..\data\flu-data

All datasets downloaded to: C:\Work\LogisticsInnovationHackathon2026\VaxFlow\data


In [40]:
import urllib.request
import dotenv 

dotenv.load_dotenv(Path("../.env"))

url0 = 'https://opend.data.go.th/get-ckan/datastore_search?resource_id=d04fc8f7-716e-437b-a835-7057c551e651'
url1 = 'https://opend.data.go.th/get-ckan/datastore_search?resource_id=2661f356-7827-4f7c-988e-339e65aaad34'
url2 = 'https://opend.data.go.th/get-ckan/datastore_search?resource_id=641f6f30-85cd-4e29-8ed3-be3ae2679a6c'
url3 = 'https://opend.data.go.th/get-ckan/datastore_search?resource_id=f4a9e9ab-156e-40ef-a51c-4f3c6533e0be'
url4 = 'https://opend.data.go.th/get-ckan/datastore_search?resource_id=1993b199-3c1a-49de-8d49-8391c8f47a7e'
url5 = 'https://opend.data.go.th/get-ckan/datastore_search?resource_id=5b4d8602-46ac-48f6-aeca-aec511e1c773'


api_key = os.getenv("OPEND_KEY")

req = urllib.request.Request(url0)
req.add_header('api-key', api_key)

fileobj = urllib.request.urlopen(req)
print(fileobj.read())

b'{"help": "https://data.go.th/api/3/action/help_show?name=datastore_search", "success": true, "result": {"include_total": true, "limit": 100, "records_format": "objects", "resource_id": "d04fc8f7-716e-437b-a835-7057c551e651", "total_estimation_threshold": null, "records": [{"_id":1,"No.":1,"Year (CE)":1977,"Year (BE)":2520,"Vaccine Name":"BCG","Vaccine Subtype":"-","Program Type":"NIP","Target Group":"Infant/Child","Target Detail":"\xe0\xb9\x80\xe0\xb8\x94\xe0\xb9\x87\xe0\xb8\x81\xe0\xb9\x81\xe0\xb8\xa3\xe0\xb8\x81\xe0\xb9\x80\xe0\xb8\x81\xe0\xb8\xb4\xe0\xb8\x94","Status":"Initiated","Description":"\xe0\xb9\x80\xe0\xb8\xa3\xe0\xb8\xb4\xe0\xb9\x88\xe0\xb8\xa1\xe0\xb9\x83\xe0\xb8\xab\xe0\xb9\x89\xe0\xb8\xa7\xe0\xb8\xb1\xe0\xb8\x84\xe0\xb8\x8b\xe0\xb8\xb5\xe0\xb8\x99 BCG \xe0\xb8\x84\xe0\xb8\xa3\xe0\xb8\xb1\xe0\xb9\x89\xe0\xb8\x87\xe0\xb9\x81\xe0\xb8\xa3\xe0\xb8\x81\xe0\xb9\x83\xe0\xb8\x99\xe0\xb9\x80\xe0\xb8\x94\xe0\xb9\x87\xe0\xb8\x81\xe0\xb8\x97\xe0\xb8\xb2\xe0\xb8\xa3\xe0\xb8\x81\xe0

## Export each OpenD request to CSV
Fetches every `url0`–`url5` request (paging through all records, since the API
defaults to only 5 per call) and writes each one to its own CSV under `../data/opend/`.

In [41]:
def fetch_records(resource_id: str, api_key: str, page_size: int = 1000) -> list[dict]:
    """Page through a CKAN datastore_search resource and return all records."""
    base = "https://opend.data.go.th/get-ckan/datastore_search"
    records, offset = [], 0
    while True:
        url = f"{base}?resource_id={resource_id}&limit={page_size}&offset={offset}"
        req = urllib.request.Request(url)
        req.add_header("api-key", api_key)
        with urllib.request.urlopen(req) as fileobj:
            result = json.loads(fileobj.read())["result"]
        batch = result["records"]
        records.extend(batch)
        offset += len(batch)
        if not batch or offset >= result.get("total", offset):
            break
    return records


OPEND_DIR = DATA_DIR / "opend"
OPEND_DIR.mkdir(parents=True, exist_ok=True)

urls = [url0, url1, url2, url3, url4, url5]
for i, url in enumerate(urls):
    resource_id = url.split("resource_id=")[1]
    records = fetch_records(resource_id, api_key)
    df = pd.DataFrame(records)
    out = OPEND_DIR / f"request_{i}_{resource_id}.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"[ok] url{i} -> {out.name}  ({df.shape[0]:,} rows x {df.shape[1]} cols)")

print(f"\nExported {len(urls)} request(s) to {OPEND_DIR.resolve()}")

[ok] url0 -> request_0_d04fc8f7-716e-437b-a835-7057c551e651.csv  (77 rows x 13 cols)
[ok] url1 -> request_1_2661f356-7827-4f7c-988e-339e65aaad34.csv  (36 rows x 4 cols)
[ok] url2 -> request_2_641f6f30-85cd-4e29-8ed3-be3ae2679a6c.csv  (924 rows x 5 cols)
[ok] url3 -> request_3_f4a9e9ab-156e-40ef-a51c-4f3c6533e0be.csv  (4,108 rows x 12 cols)
[ok] url4 -> request_4_1993b199-3c1a-49de-8d49-8391c8f47a7e.csv  (1,659 rows x 12 cols)
[ok] url5 -> request_5_5b4d8602-46ac-48f6-aeca-aec511e1c773.csv  (125,250 rows x 8 cols)

Exported 6 request(s) to C:\Work\LogisticsInnovationHackathon2026\VaxFlow\data\opend


## Load FDA Summary of Product Characteristics
Downloads the SPC `.xlsx` from the Thai FDA (catalog.fda.moph.go.th), loads it into a
DataFrame, and writes a CSV next to it under `../data/fda/`.

In [42]:
FDA_URL = "https://catalog.fda.moph.go.th/dataset/2136d98f-017a-47b9-9f12-ded4ae0fae66/resource/aecb9ffc-d092-4ce9-9da9-f2c023d3dade/download/-summary-product-characteristic.xlsx"

FDA_DIR = DATA_DIR / "fda"
FDA_DIR.mkdir(parents=True, exist_ok=True)

xlsx_path = FDA_DIR / "summary-product-characteristic.xlsx"
if not xlsx_path.exists():
    print(f"[download] {FDA_URL}")
    req = urllib.request.Request(FDA_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(xlsx_path, "wb") as f:
        f.write(resp.read())
else:
    print(f"[skip] already downloaded: {xlsx_path}")

fda_spc = pd.read_excel(xlsx_path)
csv_path = xlsx_path.with_suffix(".csv")
fda_spc.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"[ok] {xlsx_path.name} -> {csv_path.name}  ({fda_spc.shape[0]:,} rows x {fda_spc.shape[1]} cols)")
fda_spc.head()

[skip] already downloaded: ..\data\fda\summary-product-characteristic.xlsx
[ok] summary-product-characteristic.xlsx -> summary-product-characteristic.csv  (98 rows x 5 cols)


,ATC Code,ATC Name,Trade Name,Registration No.,Importer
0,J07BC02,"hepatitis A, inactivated, whole virus",AVAXIM,1C 136/41 (N),SANOFI PASTEUR LTD
1,J07BC02,"hepatitis A, inactivated, whole virus",AVAXIM 80 U PEDIATRIC,1C 109/46 (N),SANOFI PASTEUR LTD
2,J07BC02,"hepatitis A, inactivated, whole virus",HAVRIX (TM),1C 37/38 (N),GLAXOSMITHKLINE (THAILAND) LTD.
3,J07BC02,"hepatitis A, inactivated, whole virus",HEALIVE,1C 28/61 (NB),"BIONET-ASIA CO., LTD."
4,J07BC02,"hepatitis A, inactivated, whole virus",MEVAC -A,1C 8/53 (NB),"BIO GENETECH CO., LTD."


## Load FDA Human Vaccine registry
Scrapes the human-vaccine registration table from `pertento.fda.moph.go.th`
(a static HTML table) and writes it to CSV under `../data/fda/`.

In [43]:
VACCINE_URL = "https://pertento.fda.moph.go.th/FDA_SEARCH_DRUG/SEARCH_DRUG/FRM_VACCINE_HUMAN.aspx"

req = urllib.request.Request(VACCINE_URL, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as resp:
    html = resp.read().decode("utf-8", errors="replace")

# Pick the widest table on the page (the registry grid has the most columns).
tables = pd.read_html(html)
vaccine_human = max(tables, key=lambda t: t.shape[1])

out = FDA_DIR / "vaccine_human.csv"
vaccine_human.to_csv(out, index=False, encoding="utf-8-sig")
print(f"[ok] FRM_VACCINE_HUMAN -> {out.name}  ({vaccine_human.shape[0]:,} rows x {vaccine_human.shape[1]} cols)")
vaccine_human.head()

[ok] FRM_VACCINE_HUMAN -> vaccine_human.csv  (119 rows x 8 cols)


C:\Users\jetan\AppData\Local\Temp\ipykernel_29456\3682836177.py:8: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


,ลำดับ,กลุ่มตำรับยา (รหัสกลุ่มตำรับ) (ATC Classification),เลขทะเบียน (Reg. No.),ชื่อการค้าภาษาไทย (TradeName in Thai),ชื่อการค้าภาษาอังกฤษ (TradeName in Eng),ชื่อผู้รับอนุญาต (Licencee),สถานะ (Status),Unnamed: 7
0,1,Cholera vaccines(J07AE),2C 2/68 (NBC),ยูวิคอล-พลัส,Euvichol-Plus,บริษัท ไบโอจีนีเทค จำกัด,คงอยู่,รายละเอียด
1,2,Meningococcal vaccines(J07AH),2C 263/69 (NBC),-,MENFIVE,บริษัท มาสุ จำกัด,คงอยู่,รายละเอียด
2,3,"meningococcus A,C,Y,W-135, tetravalent purifie...",2C 11/66 (NBC),เม็นควอดฟี,MenQuadfi,บริษัท ซาโนฟี่-อเวนตีส (ประเทศไทย) จำกัด,คงอยู่,รายละเอียด
3,4,"meningococcus A,C,Y,W-135, tetravalent purifie...",2C 7/66 (NBC),NaN,NIMENRIX,บริษัท ไฟเซอร์ (ประเทศไทย) จำกัด,คงอยู่,รายละเอียด
4,5,"meningococcus B, multicomponent vaccine(J07AH09)",2C 5/67 (NBC),-,BEXSERO,บริษัท แกล็กโซสมิทไคล์น (ประเทศไทย) จำกัด,คงอยู่,รายละเอียด


## Compare & merge FDA datasets (full outer)
เทียบ `summary-product-characteristic.csv` (SPC: ATC + ชื่อการค้า EN + ผู้นำเข้า) กับ
`vaccine_human.csv` (ทะเบียนวัคซีน: ชื่อไทย/อังกฤษ + ผู้รับอนุญาต + สถานะ) แล้ว **merge แถวที่เหมือน/ใกล้เคียงกัน**
จับคู่ด้วย **เลขทะเบียน (Reg. No.)** ก่อน → **ชื่อการค้า (EN)** → fuzzy match
**คงแถวที่ไม่ตรงไว้ด้วย** (full outer) โดยใส่คอลัมน์ `source` = `both` / `spc_only` / `vh_only`
ผลลัพธ์: `../data/fda/vaccine_merged.csv`

In [44]:
import re
from difflib import SequenceMatcher

spc = pd.read_csv(FDA_DIR / "summary-product-characteristic.csv", encoding="utf-8-sig", dtype=str).fillna("")
vh  = pd.read_csv(FDA_DIR / "vaccine_human.csv", encoding="utf-8-sig", dtype=str).fillna("")
# header ของ SPC มี non-breaking space (\xa0) ปนใน "ATC Code" / "ATC Name"
spc.columns = [c.replace("\xa0", " ").strip() for c in spc.columns]
vh.columns  = [c.replace("\xa0", " ").strip() for c in vh.columns]

VH_ATC = "กลุ่มตำรับยา (รหัสกลุ่มตำรับ) (ATC Classification)"
VH_REG, VH_TH = "เลขทะเบียน (Reg. No.)", "ชื่อการค้าภาษาไทย (TradeName in Thai)"
VH_EN, VH_LIC, VH_ST = ("ชื่อการค้าภาษาอังกฤษ (TradeName in Eng)",
                        "ชื่อผู้รับอนุญาต (Licencee)", "สถานะ (Status)")

norm_reg  = lambda s: re.sub(r"\s+", "", re.sub(r"\(.*?\)", "", s)).upper()   # ตัด (N)/(NBC) + ช่องว่าง
norm_name = lambda s: re.sub(r"[^a-z0-9]", "", re.sub(r"\(tm\)|\(r\)|®|™", "", s.lower()))
def atc_of(s):
    m = re.search(r"\(([A-Z]\d{2}[A-Z]{0,2}\d{0,2})\)\s*$", s.strip())
    return m.group(1) if m else ""

spc["reg_k"], spc["name_k"] = spc["Registration No."].map(norm_reg), spc["Trade Name"].map(norm_name)
vh["reg_k"],  vh["name_k"]  = vh[VH_REG].map(norm_reg), vh[VH_EN].map(norm_name)

vh_by_reg  = {r.reg_k: i for i, r in vh.iterrows() if r.reg_k}
vh_by_name = {r.name_k: i for i, r in vh.iterrows() if r.name_k}
vh_names   = [(i, r.name_k) for i, r in vh.iterrows() if r.name_k]

def rec(source, match_type, s=None, v=None):
    return {"source": source, "match_type": match_type,
            "atc_code": (s["ATC Code"] if s is not None and s["ATC Code"] else (atc_of(v[VH_ATC]) if v is not None else "")),
            "atc_name": s["ATC Name"] if s is not None else "",
            "trade_name_en": s["Trade Name"] if s is not None else (v[VH_EN] if v is not None else ""),
            "trade_name_th": v[VH_TH] if v is not None else "",
            "reg_no": s["Registration No."] if s is not None else "",
            "reg_no_vh": v[VH_REG] if v is not None else "",
            "importer": s["Importer"] if s is not None else "",
            "licencee": v[VH_LIC] if v is not None else "",
            "status": v[VH_ST] if v is not None else ""}

rows, matched_vh = [], set()
for _, s in spc.iterrows():                       # ฝั่ง SPC: matched (both) หรือ spc_only
    j, mt = None, None
    if s.reg_k in vh_by_reg:          j, mt = vh_by_reg[s.reg_k], "reg_no"
    elif s.name_k in vh_by_name:      j, mt = vh_by_name[s.name_k], "trade_name"
    else:
        best = max(((SequenceMatcher(None, s.name_k, nk).ratio(), i) for i, nk in vh_names), default=(0, None))
        if best[0] >= 0.88:           j, mt = best[1], f"fuzzy({best[0]:.2f})"
    if j is None:
        rows.append(rec("spc_only", "", s=s))
    else:
        matched_vh.add(j)
        rows.append(rec("both", mt, s=s, v=vh.loc[j]))

for j, v in vh.iterrows():                         # ฝั่ง VH ที่ยังไม่ถูกจับคู่ -> vh_only
    if j not in matched_vh:
        rows.append(rec("vh_only", "", v=v))

merged = pd.DataFrame(rows)
merged.to_csv(FDA_DIR / "vaccine_merged.csv", index=False, encoding="utf-8-sig")
print(f"SPC {len(spc)} + VH {len(vh)} -> รวม {len(merged)} แถว")
print("by source:", merged["source"].value_counts().to_dict())
print("match types:", merged.loc[merged.source == 'both', 'match_type']
      .str.replace(r'\(.*\)', '', regex=True).value_counts().to_dict())
merged.head(10)

SPC 98 + VH 119 -> รวม 151 แถว
by source: {'both': 71, 'vh_only': 53, 'spc_only': 27}
match types: {'reg_no': 61, 'trade_name': 7, 'fuzzy': 3}


,source,match_type,atc_code,atc_name,trade_name_en,trade_name_th,reg_no,reg_no_vh,importer,licencee,status
0,both,reg_no,J07BC02,"hepatitis A, inactivated, whole virus",AVAXIM,-,1C 136/41 (N),1C 136/41 (N),SANOFI PASTEUR LTD,บริษัท ซาโนฟี่-อเวนตีส (ประเทศไทย) จำกัด,คงอยู่
1,both,reg_no,J07BC02,"hepatitis A, inactivated, whole virus",AVAXIM 80 U PEDIATRIC,-,1C 109/46 (N),1C 109/46 (N),SANOFI PASTEUR LTD,บริษัท ซาโนฟี่-อเวนตีส (ประเทศไทย) จำกัด,คงอยู่
2,both,reg_no,J07BC02,"hepatitis A, inactivated, whole virus",HAVRIX (TM),-,1C 37/38 (N),1C 37/38 (N),GLAXOSMITHKLINE (THAILAND) LTD.,บริษัท แกล็กโซสมิทไคล์น (ประเทศไทย) จำกัด,คงอยู่
3,both,reg_no,J07BC02,"hepatitis A, inactivated, whole virus",HEALIVE,-,1C 28/61 (NB),1C 28/61 (NB),"BIONET-ASIA CO., LTD.",บริษัท ไบโอเนท - เอเชีย จำกัด,คงอยู่
4,both,reg_no,J07BC02,"hepatitis A, inactivated, whole virus",MEVAC -A,มีแวค เอ,1C 8/53 (NB),1C 8/53 (NB),"BIO GENETECH CO., LTD.",บริษัท ไบโอจีนีเทค จำกัด,คงอยู่
5,both,reg_no,J07BC02,"hepatitis A, inactivated, whole virus",VAQTA,แวคต้า,1C 15/55 (NB),1C 15/55 (NB),"MSD (THAILAND) CO., LTD.",บริษัท เอ็มเอสดี (ประเทศไทย) จำกัด,คงอยู่
6,both,reg_no,J07BC20,combinations,TWINRIX,-,2C 24/43 (N),2C 24/43 (N),GLAXOSMITHKLINE (THAILAND) LTD.,บริษัท แกล็กโซสมิทไคล์น (ประเทศไทย) จำกัด,คงอยู่
7,both,reg_no,J07BD01,"measles, live attenuated",MEASLES VACCINE,มีเซิล วัคซีน,1C 1/59 (B),1C 1/59 (B),"BIONET-ASIA CO., LTD.",บริษัท ไบโอเนท - เอเชีย จำกัด,คงอยู่
8,both,fuzzy(0.89),J07BD01,"measles, live attenuated","MEASLES VACCINE, LIVE, ATTENUATED (FREEZE-DRIED)",-,1C 16/61 (B),2C 10/66 (B),"MASU CO., LTD.",บริษัท มาสุ จำกัด,คงอยู่
9,both,reg_no,J07BD52,"measles, combinations with mumps and rubella, ...","MEASLES,MUMPS AND RUBELLA VACCINE LIVE, ATTENU...",-,2C 46/48 (B),2C 46/48,"MASU CO., LTD.",บริษัท มาสุ จำกัด,คงอยู่


## Enrich with cold-chain storage → `vaccine_merged_with_storage.csv`
เติม **คุณลักษณะการจัดเก็บ (cold-chain)** ให้แต่ละวัคซีนตาม ATC class + ชื่อการค้า เพื่อให้เป็น
**product master พร้อมใช้ในโปรเจกต์** (ป้อน state machine: `DEEP_FROZEN → THAWED → OPENED`)

| field | ความหมาย |
|-------|----------|
| `vax_type` | `mRNA` (ต้องแช่แข็งจัด) / `MULTI_DOSE` |
| `deep_frozen_life_days` | อายุสถานะแช่แข็งจัด (mRNA เท่านั้น · 0 = ไม่ต้องแช่แข็งจัด) |
| `thawed_life_days` | อายุที่ 2–8°C |
| `open_life_hours` | อายุหลังเปิดขวด (lyophilised 6 ชม. · liquid multidose ตาม WHO MDVP ≤ 28 วัน) |
| `doses_per_vial` | จำนวนโดสต่อขวด |

> ⚠️ ค่าจัดเก็บเป็น **ค่าประมาณอ้างอิง WHO Multi-Dose Vial Policy / SmPC** — ตอนใช้จริงควร override จากฉลากของแต่ละทะเบียน

In [45]:
merged = pd.read_csv(FDA_DIR / "vaccine_merged.csv", encoding="utf-8-sig", dtype=str).fillna("")

def storage_profile(atc_code: str, name: str) -> dict:
    """กำหนดคุณลักษณะ cold-chain ตาม ATC class + ชื่อการค้า (ค่าประมาณ WHO MDVP / SmPC)."""
    t = (atc_code + " " + name).lower()
    a = atc_code.upper()
    # 1) mRNA — ต้องแช่แข็งจัด แล้วนับถอยหลัง 30 วันเมื่อย้ายมา 2-8°C
    if any(k in t for k in ("mrna", "comirnaty", "spikevax")):
        return dict(vax_type="mRNA", storage_temp="-90..-60°C → 2-8°C",
                    deep_frozen_life_days=365, thawed_life_days=30, open_life_hours=6, doses_per_vial=6)
    # 2) live/lyophilised (ผงแห้ง) ต้องผสมก่อนใช้ → ทิ้งใน 6 ชม.
    if a[:5] == "J07AN" or any(k in t for k in ("bcg", "measles", "mumps", "rubella", "mmr",
                                                "varicella", "zoster", "yellow")):
        return dict(vax_type="MULTI_DOSE", storage_temp="2-8°C (lyophilised)",
                    deep_frozen_life_days=0, thawed_life_days=730, open_life_hours=6, doses_per_vial=10)
    # 3) oral polio / liquid multidose → WHO MDVP ใช้ได้ถึง 28 วัน
    if a[:5] == "J07BF" or "opv" in t or "oral polio" in t:
        return dict(vax_type="MULTI_DOSE", storage_temp="2-8°C (liquid, oral)",
                    deep_frozen_life_days=0, thawed_life_days=730, open_life_hours=28 * 24, doses_per_vial=20)
    # 4) liquid multidose (bacterial/combo/hepatitis) → ≤ 28 วันหลังเปิด
    if a[:4] in ("J07A", "J07C") or a[:5] == "J07BC" or any(k in t for k in (
            "hepatitis b", "dtp", "pentavalent", "tetanus", "diphtheria",
            "pertussis", "cholera", "typhoid", "pneumo", "menin")):
        return dict(vax_type="MULTI_DOSE", storage_temp="2-8°C (liquid)",
                    deep_frozen_life_days=0, thawed_life_days=730, open_life_hours=28 * 24, doses_per_vial=10)
    # 5) default — single-dose prefilled (influenza/HPV/rabies/IPV ฯลฯ) เปิดแล้วใช้ทันที
    return dict(vax_type="MULTI_DOSE", storage_temp="2-8°C (single-dose)",
                deep_frozen_life_days=0, thawed_life_days=730, open_life_hours=6, doses_per_vial=1)

prof = merged.apply(lambda r: storage_profile(r["atc_code"], r["trade_name_en"]),
                    axis=1, result_type="expand")
out = pd.concat([merged, prof], axis=1)
# product_id ใช้เป็น key ในโปรเจกต์ (VAX_<ATC>_<seq>)
out.insert(0, "product_id", [f"VAX_{(a or 'NA').replace(' ', '')}_{i:03d}"
                             for i, a in enumerate(out["atc_code"], 1)])

out.to_csv(FDA_DIR / "vaccine_merged_with_storage.csv", index=False, encoding="utf-8-sig")
print(f"[ok] vaccine_merged_with_storage.csv: {out.shape[0]} rows x {out.shape[1]} cols")
print("vax_type:", out["vax_type"].value_counts().to_dict())
print("open_life_hours:", out["open_life_hours"].value_counts().to_dict())
out[["product_id", "atc_code", "trade_name_en", "trade_name_th", "vax_type",
     "deep_frozen_life_days", "thawed_life_days", "open_life_hours", "doses_per_vial"]].head(10)

[ok] vaccine_merged_with_storage.csv: 151 rows x 18 cols
vax_type: {'MULTI_DOSE': 146, 'mRNA': 5}
open_life_hours: {672: 77, 6: 74}


,product_id,atc_code,trade_name_en,trade_name_th,vax_type,deep_frozen_life_days,thawed_life_days,open_life_hours,doses_per_vial
0,VAX_J07BC02_001,J07BC02,AVAXIM,-,MULTI_DOSE,0,730,672,10
1,VAX_J07BC02_002,J07BC02,AVAXIM 80 U PEDIATRIC,-,MULTI_DOSE,0,730,672,10
2,VAX_J07BC02_003,J07BC02,HAVRIX (TM),-,MULTI_DOSE,0,730,672,10
3,VAX_J07BC02_004,J07BC02,HEALIVE,-,MULTI_DOSE,0,730,672,10
4,VAX_J07BC02_005,J07BC02,MEVAC -A,มีแวค เอ,MULTI_DOSE,0,730,672,10
5,VAX_J07BC02_006,J07BC02,VAQTA,แวคต้า,MULTI_DOSE,0,730,672,10
6,VAX_J07BC20_007,J07BC20,TWINRIX,-,MULTI_DOSE,0,730,672,10
7,VAX_J07BD01_008,J07BD01,MEASLES VACCINE,มีเซิล วัคซีน,MULTI_DOSE,0,730,6,10
8,VAX_J07BD01_009,J07BD01,"MEASLES VACCINE, LIVE, ATTENUATED (FREEZE-DRIED)",-,MULTI_DOSE,0,730,6,10
9,VAX_J07BD52_010,J07BD52,"MEASLES,MUMPS AND RUBELLA VACCINE LIVE, ATTENU...",-,MULTI_DOSE,0,730,6,10


## Convert all files to CSV
Converts every non-CSV file under `../data/` (`.xlsx`, `.json`, `.js`) into a `.csv`
sitting next to the original.

In [46]:
import re
import json5


def js_to_records(text: str):
    """Extract the array literal from a JS file (e.g. `const x = [ ... ];`) and parse it."""
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1:
        raise ValueError("No array literal found in JS file")
    return json5.loads(text[start : end + 1])


def to_dataframe(fp: Path) -> pd.DataFrame:
    suffix = fp.suffix.lower()
    if suffix in (".xlsx", ".xls"):
        return pd.read_excel(fp)
    if suffix == ".json":
        with open(fp, encoding="utf-8") as f:
            data = json.load(f)
        return pd.json_normalize(data)
    if suffix == ".js":
        return pd.json_normalize(js_to_records(fp.read_text(encoding="utf-8")))
    raise ValueError(f"Unsupported file type: {fp.suffix}")


converted = []
for fp in sorted(DATA_DIR.rglob("*")):
    if not fp.is_file() or fp.suffix.lower() not in (".xlsx", ".xls", ".json", ".js"):
        continue
    out = fp.with_suffix(".csv")
    try:
        df = to_dataframe(fp)
        df.to_csv(out, index=False)
        converted.append(out)
        print(f"[ok] {fp.relative_to(DATA_DIR)} -> {out.name}  ({df.shape[0]:,} rows x {df.shape[1]} cols)")
    except Exception as e:
        print(f"[fail] {fp.relative_to(DATA_DIR)}: {e}")

print(f"\nConverted {len(converted)} file(s) to CSV.")

[ok] fda\summary-product-characteristic.xlsx -> summary-product-characteristic.csv  (98 rows x 5 cols)
[ok] inventory-data-for-pharmacy-website-in-json-format\data.js -> data.csv  (93 rows x 12 cols)
[ok] pharmaceutical-supply-chain-optimization\Pharmaceutical Supply Chain Optimization.xlsx -> Pharmaceutical Supply Chain Optimization.csv  (100,000 rows x 4 cols)

Converted 3 file(s) to CSV.


In [47]:
def load_file(fp: Path):
    if fp.suffix.lower() == ".csv":
        return pd.read_csv(fp)
    if fp.suffix.lower() == ".json":
        with open(fp, encoding="utf-8") as f:
            data = json.load(f)
        return pd.DataFrame(data) if isinstance(data, list) else data
    return None


datasets: dict[str, dict[str, pd.DataFrame]] = {}

for folder in sorted(DATA_DIR.iterdir()):
    if not folder.is_dir():
        continue
    print(f"\n=== {folder.name} ===")
    datasets[folder.name] = {}
    for fp in sorted(folder.rglob("*")):
        if not fp.is_file() or fp.suffix.lower() not in (".csv", ".json"):
            continue
        result = load_file(fp)
        rel = str(fp.relative_to(folder))
        if isinstance(result, pd.DataFrame):
            datasets[folder.name][rel] = result
            print(f"  {rel}: {result.shape[0]:,} rows × {result.shape[1]} cols")
        else:
            print(f"  {rel}: nested JSON — inspect manually")


=== covid-world-vaccination-progress ===
  country_vaccinations.csv: 86,512 rows × 15 cols
  country_vaccinations_by_manufacturer.csv: 35,623 rows × 4 cols

=== external ===

=== fda ===
  summary-product-characteristic.csv: 98 rows × 5 cols
  vaccine_human.csv: 119 rows × 8 cols
  vaccine_merged.csv: 151 rows × 11 cols
  vaccine_merged_with_storage.csv: 151 rows × 18 cols

=== flu-data ===
  H1N1_Flu_Vaccines.csv: 26,707 rows × 38 cols

=== hospitals ===
  hospital_master.csv: 13 rows × 5 cols

=== inventory-data-for-pharmacy-website-in-json-format ===
  data.csv: 93 rows × 12 cols

=== opend ===
  request_0_d04fc8f7-716e-437b-a835-7057c551e651.csv: 77 rows × 13 cols
  request_1_2661f356-7827-4f7c-988e-339e65aaad34.csv: 36 rows × 4 cols
  request_2_641f6f30-85cd-4e29-8ed3-be3ae2679a6c.csv: 924 rows × 5 cols
  request_3_f4a9e9ab-156e-40ef-a51c-4f3c6533e0be.csv: 4,108 rows × 12 cols
  request_4_1993b199-3c1a-49de-8d49-8391c8f47a7e.csv: 1,659 rows × 12 cols
  request_5_5b4d8602-46ac-48f

## Quick preview

In [48]:
for folder, files in datasets.items():
    for fname, df in files.items():
        print(f"\n--- {folder} / {fname} ---")
        display(df.head(3))


--- covid-world-vaccination-progress / country_vaccinations.csv ---


,country,iso_code,date,total_vaccinations,people_vaccinated,people_fully_vaccinated,daily_vaccinations_raw,daily_vaccinations,total_vaccinations_per_hundred,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,daily_vaccinations_per_million,vaccines,source_name,source_website
0,Afghanistan,AFG,2021-02-22,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/
1,Afghanistan,AFG,2021-02-23,NaN,NaN,NaN,NaN,1367.0,NaN,NaN,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/
2,Afghanistan,AFG,2021-02-24,NaN,NaN,NaN,NaN,1367.0,NaN,NaN,NaN,34.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",World Health Organization,https://covid19.who.int/



--- covid-world-vaccination-progress / country_vaccinations_by_manufacturer.csv ---


,location,date,vaccine,total_vaccinations
0,Argentina,2020-12-29,Moderna,2
1,Argentina,2020-12-29,Oxford/AstraZeneca,3
2,Argentina,2020-12-29,Sinopharm/Beijing,1



--- fda / summary-product-characteristic.csv ---


...

,hospital_id,transport_rate
0,HOSP_001,15.41
1,HOSP_002,9.93
2,HOSP_003,19.26



--- vaccine / vaccine_product.csv ---


,product_id,name,type,doses_per_vial,deep_frozen_life_days,thawed_life_days,open_life_hours
0,VAX_J07BC02_001,AVAXIM,MULTI_DOSE,10,0,730,672
1,VAX_J07BC02_002,AVAXIM 80 U PEDIATRIC,MULTI_DOSE,10,0,730,672
2,VAX_J07BC02_003,HAVRIX (TM),MULTI_DOSE,10,0,730,672



--- vaccine / vaccine_vial.csv ---


,vial_id,lot_id,product_id,hospital_id,state,state_since,doses_remaining,label_expiry,effective_expiry
0,VIAL_000001,LOT_001_202606,VAX_J07BC02_001,HOSP_001,THAWED,2026-06-16T08:00:00+07:00,10,2026-12-05,2026-12-05T00:00:00+07:00
1,VIAL_000002,LOT_001_202606,VAX_J07BC02_001,HOSP_001,THAWED,2026-06-08T08:00:00+07:00,10,2027-02-13,2027-02-13T00:00:00+07:00
2,VIAL_000003,LOT_001_202606,VAX_J07BC02_001,HOSP_001,THAWED,2026-06-07T08:00:00+07:00,10,2027-01-17,2027-01-17T00:00:00+07:00
